In [1]:
import Pkg

In [2]:
Pkg.activate(".")
Pkg.add("ThreadPinning")
Pkg.add("KrylovKit")
Pkg.add("BenchmarkTools")
Pkg.add("ProfileCanvas")
Pkg.instantiate()

  Activating project at `~/Documents/KrylovKitBenchmark`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`


In [3]:
using Random
using KrylovKit: expintegrator, Arnoldi
using BenchmarkTools
using ProfileCanvas
using LinearAlgebra: norm

In [4]:
using ThreadPinning
pinthreads(:cores)
threadinfo()

Hostname: 	cuny
CPU(s): 	2 x Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz
CPU target: 	cascadelake
Cores: 		32 (64 CPU-threads due to 2-way SMT)
NUMA domains: 	2 (16 cores each)

Julia threads: 	8

CPU socket 1
  0,32, 1,33, 2,34, 3,35, 4,36, 5,37, 6,38, 7,39, 
  8,40, 9,41, 10,42, 11,43, 12,44, 13,45, 14,46, 15,47

CPU socket 2
  16,48, 17,49, 18,50, 19,51, 20,52, 21,53, 22,54, 23,55, 
  24,56, 25,57, 26,58, 27,59, 28,60, 29,61, 30,62, 31,63


# = Julia thread, # = Julia thread on HT, # = >1 Julia thread

(Mapping: 1 => 0, 2 => 1, 3 => 2, 4 => 3, 5 => 4, ...)


In [5]:
filter(p -> contains(p[1], "THREAD"), ENV)

Dict{String, String} with 6 entries:
  "OPENBLAS_NUM_THREADS"   => "1"
  "VECLIB_MAXIMUM_THREADS" => "1"
  "OMP_NUM_THREADS"        => "1"
  "NUMEXPR_NUM_THREADS"    => "1"
  "MKL_NUM_THREADS"        => "1"
  "JULIA_NUM_THREADS"      => "8"

In [6]:
N = 100

100

In [7]:
"""Random complex matrix of dimension `N` with a spectral radius of approximately `ρ`."""
function random_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    H = ρ * (X + Y * 1im) / √2
    return H
end

function random_hermitian_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    Z = (X + Y * 1im) / √2
    H = ρ * (Z + Z') / (2 * √2)
    return H
end


"""Random normalized complex vector of dimension `N`"""
function random_state_vector(N=N; rng=Random.GLOBAL_RNG)
    Ψ = rand(rng, N) .* exp.((2π * im) .* rand(rng, N))
    Ψ ./= norm(Ψ)
    return Ψ
end

random_state_vector

In [8]:
struct Trajectory
    initial_state::Vector{ComplexF64}
    H::Matrix{ComplexF64}
    dt::Vector{Float64}
end

function Trajectory(;initial_state, H, nt)
    dt = rand(nt)
    N = length(initial_state)
    @assert size(H) == (N, N)
    Trajectory(initial_state, H, dt)
end

Trajectory

In [9]:
function propagate_traj(traj::Trajectory)
    # even if traj.H is Hermitian, we still use Arnoldi.
    # This propagation method is intended for non-Hermitian generators,
    # using a general matrix screws up the benchmark because the norm
    # of Ψ explodes.
    alg = Arnoldi()
    Ψ = traj.initial_state
    numops = 0
    for dt in traj.dt
        Ψ, info = expintegrator(traj.H, -1im * dt, (Ψ, ), alg)
        numops += info.numops
    end
    return numops
end

propagate_traj (generic function with 1 method)

In [10]:
traj100 = Trajectory(initial_state=random_state_vector(), H=random_hermitian_matrix(), nt=100);
propagate_traj(traj100)

3100

In [11]:
@benchmark propagate_traj(traj100)

BenchmarkTools.Trial: 116 samples with 1 evaluation.
 Range (min … max):  40.238 ms … 61.211 ms  ┊ GC (min … max): 0.00% … 21.97%
 Time  (median):     42.613 ms              ┊ GC (median):    3.10%
 Time  (mean ± σ):   43.105 ms ±  2.537 ms  ┊ GC (mean ± σ):  4.05% ±  3.58%

      ▂▂▁▅█▂▄ ▂ ▅▇▂                                            
  █▅▅▅███████▅██████▆▆▆▃▃▁▁▃▁▃▁▁▁▁▁▁▃▁▃▁▁▁▃▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▃ ▃
  40.2 ms         Histogram: frequency by time          53 ms <

 Memory estimate: 19.69 MiB, allocs estimate: 16100.

In [12]:
Base.GC.enable(false)
@benchmark propagate_traj(traj100)

BenchmarkTools.Trial: 98 samples with 1 evaluation.
 Range (min … max):  49.278 ms … 59.411 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     50.606 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   51.114 ms ±  1.619 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

      ▄▄▅▅█ ▃▄                                                 
  ▃▁▁▅████████▇█▁▅▃▃▃▆▃▃▃▁▃▁▁▃▁▃▅▃▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▃ ▁
  49.3 ms         Histogram: frequency by time        57.9 ms <

 Memory estimate: 19.69 MiB, allocs estimate: 16100.

In [13]:
Base.GC.enable(true)

false

In [14]:
@profview propagate_traj(traj100)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("3" => ProfileCanvas.ProfileFrame("root", "", "", 0, 32, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 32, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 32, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 31, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1004, 3, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 142, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1315, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 916, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("length", "essentials.jl", "./essentials.jl", 11, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 144, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame(">", "operators.jl", "./operators.jl", 379, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("<", "int.jl", "./int.jl", 513, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])]), ProfileCanvas.ProfileFrame("multiq_check_empty", "partr.jl", "./partr.jl", 186, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 917, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])]), ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1010, 1, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1004, 1, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 150, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trylock", "locks-mt.jl", "./locks-mt.jl", 53, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])])])])]), "4" => ProfileCanvas.ProfileFrame("root", "", "", 0, 32, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 32, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 32, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 32, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1004, 1, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 142, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1315, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 916, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("length", "essentials.jl", "./essentials.jl", 11, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])])]), ProfileCanvas.ProfileFrame("multiq_check_empty", "partr.jl", "./partr.jl", 186, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 916, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])])])]), "1" => ProfileCanvas.ProfileFrame("root", "", "", 0, 32, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#15", "eventloop.jl", "/home

In [15]:
traj1000 = Trajectory(initial_state=random_state_vector(), H=random_hermitian_matrix(), nt=1000);
propagate_traj(traj1000)

31000

In [16]:
@benchmark propagate_traj($traj1000)

BenchmarkTools.Trial: 12 samples with 1 evaluation.
 Range (min … max):  422.072 ms … 441.861 ms  ┊ GC (min … max): 1.92% … 6.16%
 Time  (median):     426.856 ms               ┊ GC (median):    3.98%
 Time  (mean ± σ):   428.178 ms ±   6.042 ms  ┊ GC (mean ± σ):  4.00% ± 0.91%

  ▁ █ ▁    ▁  ▁   ▁ ▁    ▁     ▁              ▁               ▁  
  █▁█▁█▁▁▁▁█▁▁█▁▁▁█▁█▁▁▁▁█▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  422 ms           Histogram: frequency by time          442 ms <

 Memory estimate: 196.87 MiB, allocs estimate: 161008.

In [17]:
@profview propagate_traj(traj1000)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("3" => ProfileCanvas.ProfileFrame("root", "", "", 0, 231, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 225, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 225, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 223, missing, 0x11, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1004, 32, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 140, 11, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("cong", "partr.jl", "./partr.jl", 23, 11, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 143, 6, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1315, 4, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 916, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 917, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("to_indices", "indices.jl", "./indices.jl", 365, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("to_indices", "indices.jl", "./indices.jl", 368, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("to_index", "indices.jl", "./indices.jl", 292, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("to_index", "indices.jl", "./indices.jl", 307, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("convert", "number.jl", "./number.jl", 7, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("Int64", "boot.jl", "./boot.jl", 892, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("toInt64", "boot.jl", "./boot.jl", 816, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])])])])])]), ProfileCanvas.ProfileFrame("getproperty", "Base.jl", "./Base.jl", 49, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 141, 5, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("cong", "partr.jl", "./partr.jl", 23, 5, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 0, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 160, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 142, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1315, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 917, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 144, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame(">", "operators.jl", "./operators.jl", 379, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("<", "int.jl", "./int.jl", 513, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 138, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("mult